# M03-L04 — Avaliação de Modelos, Comparação e Reprodutibilidade

**Duração estimada:** 2 horas

## Antes de começar

1. Abra este notebook no Google Colab.
2. Vá em **Arquivo > Salvar uma cópia no Drive** para ter sua própria versão editável.
3. Execute as células **em ordem**, de cima para baixo. Pular células pode gerar erros que não fazem sentido.
4. Sempre que encontrar `# TODO`, é a sua vez de escrever código ou completar uma célula de Markdown.

Nesta lição você vai atuar como um **Cientista de Dados de Diagnóstico**: você não vai escrever a lógica de treinamento do zero (isso você já fez na M03-L03) — você vai **executar, interpretar criticamente e registrar diagnósticos** sobre os modelos que já treinou.

## 1. Retomada rápida da M03-L03

Nas lições anteriores você:

* preparou e salvou `X_treino.csv`, `X_teste.csv`, `y_treino.csv` e `y_teste.csv` na M03-L02;
* reabriu esses quatro arquivos na M03-L03;
* treinou três classificadores com `fit()` — `LogisticRegression`, `DecisionTreeClassifier` e `KNeighborsClassifier` — e gerou previsões com `predict()`.

Nesta lição vamos carregar **a mesma divisão de treino e teste produzida na M03-L02**, retreinar rapidamente os modelos para que o notebook seja independente da execução da L03 e focar na etapa que faltou: **avaliar e comparar** os resultados de forma quantitativa.

In [1]:
# Bibliotecas que vamos usar nesta lição
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

print("Bibliotecas importadas com sucesso.")

Bibliotecas importadas com sucesso.


## 2. Atividade 1 — Carregando e vinculando o trabalho prévio

Vamos reabrir exatamente os quatro arquivos gerados ao final da M03-L02:

- `X_treino.csv`
- `X_teste.csv`
- `y_treino.csv`
- `y_teste.csv`

A divisão treino/teste **já foi realizada na M03-L02**, portanto não devemos chamar `train_test_split()` novamente nesta etapa. Assim, a avaliação desta lição usa o mesmo conjunto de teste empregado na M03-L03.

In [2]:
X_train = pd.read_csv("X_treino.csv")
X_test = pd.read_csv("X_teste.csv")

y_train = pd.read_csv("y_treino.csv")["especie"]
y_test = pd.read_csv("y_teste.csv")["especie"]

print("Arquivos da M03-L02 carregados com sucesso.")

Arquivos da M03-L02 carregados com sucesso.


In [3]:
print("Formato de X_train:", X_train.shape)
print("Formato de y_train:", y_train.shape)
print("Formato de X_test:", X_test.shape)
print("Formato de y_test:", y_test.shape)

print("\nTreino alinhado:", len(X_train) == len(y_train))
print("Teste alinhado:", len(X_test) == len(y_test))

Formato de X_train: (120, 4)
Formato de y_train: (120,)
Formato de X_test: (30, 4)
Formato de y_test: (30,)

Treino alinhado: True
Teste alinhado: True


### Exercício de análise

Use `value_counts(normalize=True)` para observar a proporção de cada espécie no conjunto de teste.

O `iris.csv` original possui a mesma quantidade de amostras de `setosa`, `versicolor` e `virginica`. Porém, a divisão feita na M03-L02 usou `train_test_split()` **sem `stratify`**. Como o conjunto de teste é pequeno, suas proporções não precisam ficar exatamente em 1/3 para cada classe.

`normalize=True` faz o `value_counts()` devolver proporções (frações entre 0 e 1) em vez de contagens absolutas.

In [9]:
# TODO: calcule as proporções de classe em y_test usando value_counts(normalize=True)
proporcoes_teste = value_counts(normalize=True)  # substitua None pelo código correto

proporcoes_teste

NameError: name 'value_counts' is not defined

**Reflexão rápida:** as três classes ficaram igualmente representadas no conjunto de teste? Se houver diferença, isso significa que o dataset original deixou de ser balanceado ou é apenas um efeito da divisão aleatória? Escreva uma frase curta com sua conclusão.

*(Escreva sua resposta aqui.)*

## 3. Atividade 2 — Instanciação, treinamento e inferência de referência

Vamos retreinar os três modelos principais da L03, mais a SVM como desafio extra. Para garantir que os resultados desta seção sejam sempre os mesmos entre execuções, usamos `random_state=42` em todo componente estocástico — exatamente como você já viu, sem explicação formal, na M03-L02 e na M03-L03.

In [5]:
# Modelos principais (mesmos da M03-L03)
modelo_logistica = LogisticRegression(max_iter=1000)
modelo_arvore = DecisionTreeClassifier(random_state=42)
modelo_knn = KNeighborsClassifier(n_neighbors=5)

modelo_logistica.fit(X_train, y_train)
modelo_arvore.fit(X_train, y_train)
modelo_knn.fit(X_train, y_train)

pred_logistica = modelo_logistica.predict(X_test)
pred_arvore = modelo_arvore.predict(X_test)
pred_knn = modelo_knn.predict(X_test)

print("Os três modelos foram treinados e já geraram previsões sobre X_test.")

Os três modelos foram treinados e já geraram previsões sobre X_test.


In [6]:
# Desafio extra: SVM sobre os dados padronizados (mesmo padrão de padronização usado para o KNN na L03)
scaler = StandardScaler()
X_train_padronizado = scaler.fit_transform(X_train)
X_test_padronizado = scaler.transform(X_test)

from sklearn.svm import SVC

modelo_svm = SVC(random_state=42)
modelo_svm.fit(X_train_padronizado, y_train)
pred_svm = modelo_svm.predict(X_test_padronizado)

print("SVM (desafio extra) treinada e com previsões geradas.")

SVM (desafio extra) treinada e com previsões geradas.


## 4. Exemplo guiado 1 — Matriz de confusão da Regressão Logística

Vamos gerar a matriz de confusão do primeiro modelo e ler a diagonal principal juntos.

In [7]:
matriz_logistica = confusion_matrix(y_test, pred_logistica)
print("Classes na ordem:", sorted(y_test.unique()))
print(matriz_logistica)

Classes na ordem: ['setosa', 'versicolor', 'virginica']
[[11  0  0]
 [ 0 11  0]
 [ 0  0  8]]


**Leitura assistida:** a diagonal principal (de cima-esquerda para baixo-direita) mostra os acertos de cada espécie, na mesma ordem em que as classes aparecem — por padrão, ordem alfabética: `setosa`, `versicolor`, `virginica`. Qualquer número fora da diagonal é uma confusão entre duas espécies. Compare o total da diagonal com o total de amostras do teste (`X_test.shape[0]`) para ter uma primeira noção da acurácia geral.

## 5. Atividade 3 — Geração e desconstrução de matrizes de confusão

Agora é sua vez. Gere as matrizes de confusão da Árvore de Decisão e do KNN.

In [8]:
# TODO: gere a matriz de confusão da Árvore de Decisão
matriz_arvore = None

# TODO: gere a matriz de confusão do KNN
matriz_knn = None

print("Árvore de Decisão:")
print(matriz_arvore)
print("
KNN:")
print(matriz_knn)

SyntaxError: unterminated string literal (detected at line 9) (3114480664.py, line 9)

### Diagnóstico em Markdown (edite esta célula)

Responda com base nas duas matrizes que você acabou de gerar:

* Quais flores foram **perfeitamente** classificadas (nenhum erro naquela espécie), em cada modelo?
* Houve alguma confusão específica entre `versicolor` e `virginica`? Em qual direção (real → previsto)?
* Comparando as duas matrizes, qual modelo parece concentrar mais erros fora da diagonal principal?

*(Escreva seu diagnóstico aqui, um parágrafo curto por pergunta.)*

## 6. Exemplo guiado 2 — `classification_report` de um único modelo

Vamos gerar e ler juntos o relatório completo da Árvore de Decisão.

In [ ]:
print(classification_report(y_test, pred_arvore))

**Leitura assistida célula por célula:**

* cada linha de espécie mostra `precision`, `recall`, `f1-score` e `support` (quantas amostras reais daquela espécie existem no teste) — o `support` deve ser coerente com as contagens observadas na Atividade 1;
* `macro avg` é a média simples entre as três espécies — cada classe recebe o mesmo peso;
* `weighted avg` é a média ponderada pelo `support` — classes com mais exemplos no conjunto de teste têm maior influência nesse valor.

Como a divisão da M03-L02 não foi estratificada, os `support` das três classes podem ser um pouco diferentes mesmo que o dataset completo seja balanceado.

## 7. Atividade 4 — O relatório consolidado de classificação

Gere o `classification_report` para os quatro modelos (Regressão Logística, Árvore de Decisão, KNN e SVM — desafio extra) e monte uma tabela comparativa simples.

In [ ]:
# TODO: imprima o classification_report da Regressão Logística


In [ ]:
# TODO: imprima o classification_report do KNN


In [ ]:
# TODO: imprima o classification_report da SVM (desafio extra)


In [ ]:
# Monte aqui uma tabela comparativa simples com a acurácia geral de cada modelo.
# Dica: accuracy_score(y_test, y_pred) devolve a acurácia geral de um modelo.

tabela_comparativa = pd.DataFrame({
    "modelo": ["Regressão Logística", "Árvore de Decisão", "KNN"],  # adicione "SVM" se for fazer o desafio extra
    "acuracia": [
        accuracy_score(y_test, pred_logistica),
        accuracy_score(y_test, pred_arvore),
        accuracy_score(y_test, pred_knn),
        # TODO (desafio extra): adicione accuracy_score(y_test, pred_svm) à lista acima e o nome "SVM" à lista de modelos
    ]
})

tabela_comparativa

### Conclusão justificada (edite esta célula)

Considere um cenário hipotético em que um **falso positivo na classe `virginica`** tem custo alto para a equipe do LIPAI — isto é, afirmar que uma flor é `virginica` quando ela pertence a outra espécie seria um erro especialmente indesejado.

* Qual modelo você escolheria nesse cenário?
* Sua escolha se baseou apenas na acurácia geral ou você também observou a **precisão da classe `virginica`**?
* A revocação (`recall`) de `virginica` revela algum custo ou troca importante nessa escolha?

Justifique com os números obtidos no seu próprio `classification_report`.

*(Escreva sua conclusão aqui, em 3 a 5 frases.)*

## 8. Exemplo guiado 3 — Uma execução sem `random_state`

Vamos comparar, uma única vez, a acurácia de uma divisão treino/teste **com** semente fixa e **sem** semente fixa.

In [ ]:
# Com random_state fixo (reprodutível)
X_treino_fixo, X_teste_fixo, y_treino_fixo, y_teste_fixo = train_test_split(
    pd.concat([X_train, X_test]),
    pd.concat([y_train, y_test]),
    test_size=0.3,
    random_state=42
)

arvore_fixa = DecisionTreeClassifier(random_state=42)
arvore_fixa.fit(X_treino_fixo, y_treino_fixo)
acuracia_fixa = accuracy_score(y_teste_fixo, arvore_fixa.predict(X_teste_fixo))

# Sem random_state (random_state=None): cada execução pode sortear uma divisão diferente
X_treino_livre, X_teste_livre, y_treino_livre, y_teste_livre = train_test_split(
    pd.concat([X_train, X_test]), pd.concat([y_train, y_test]),
    test_size=0.3, random_state=None
)

arvore_livre = DecisionTreeClassifier(random_state=None)
arvore_livre.fit(X_treino_livre, y_treino_livre)
acuracia_livre = accuracy_score(y_teste_livre, arvore_livre.predict(X_teste_livre))

print(f"Acurácia com random_state fixo (42): {acuracia_fixa:.4f}")
print(f"Acurácia sem random_state (None), nesta execução: {acuracia_livre:.4f}")
print("Execute esta célula de novo (Ctrl+Enter) e compare os dois valores de novo.")

**Observação:** rode a célula acima mais de uma vez. O valor "com `random_state` fixo (42)" deve se repetir sempre igual. O valor "sem `random_state`" tende a mudar a cada execução — essa é a instabilidade que vamos investigar de forma mais sistemática na próxima atividade.

## 9. Atividade 5 — Provocando a instabilidade

O script abaixo já está pronto: ele repete a divisão treino/teste **10 vezes sem `random_state`**, treinando uma Árvore de Decisão a cada repetição e guardando a acurácia obtida.

In [ ]:
X_completo = pd.concat([X_train, X_test])
y_completo = pd.concat([y_train, y_test])

acuracias_sem_semente = []

for repeticao in range(10):
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_completo, y_completo, test_size=0.3, random_state=None
    )
    arvore_temp = DecisionTreeClassifier(random_state=None)
    arvore_temp.fit(X_tr, y_tr)
    acuracia = accuracy_score(y_te, arvore_temp.predict(X_te))
    acuracias_sem_semente.append(acuracia)
    print(f"Repetição {repeticao + 1}: acurácia = {acuracia:.4f}")

print()
print(f"Menor acurácia observada: {min(acuracias_sem_semente):.4f}")
print(f"Maior acurácia observada: {max(acuracias_sem_semente):.4f}")
print(f"Diferença entre a maior e a menor: {max(acuracias_sem_semente) - min(acuracias_sem_semente):.4f}")

### Reflexão crítica (edite esta célula)

* A acurácia variou entre as 10 repetições, mesmo usando sempre o mesmo dataset e o mesmo tipo de modelo?
* Se você tivesse ajustado um hiperparâmetro do modelo e a acurácia tivesse subido um pouco (por exemplo, de 0.88 para 0.91) **sem** `random_state` fixo, você conseguiria afirmar com segurança que a melhoria veio do ajuste, e não apenas de uma divisão de dados favorável? Por quê?

*(Escreva sua reflexão aqui.)*

## 10. Exemplo de erro comum

A célula abaixo contém um erro **de propósito**: ela compara as previsões feitas sobre `X_test` com o gabarito de **treino** (`y_train`), em vez do gabarito de teste (`y_test`).

In [ ]:
# Célula com erro proposital — execute e observe o que acontece
try:
    print(classification_report(y_train, pred_arvore))
except Exception as erro:
    print("Ocorreu um erro:")
    print(erro)

Repare que o erro acontece porque `y_train` e `pred_arvore` (que foi calculado sobre `X_test`) têm tamanhos diferentes — são conjuntos diferentes, e a função de métrica não consegue cruzar linha a linha coisas de tamanhos distintos. Esse é um dos erros mais comuns ao avaliar modelos: confundir qual conjunto (treino ou teste) deve ser usado como gabarito em cada etapa.

## 11. Exercício de correção de erro

Corrija a célula anterior: use o gabarito e a previsão corretos para avaliar o desempenho da Árvore de Decisão sobre o conjunto de **teste**.

In [ ]:
# TODO: corrija a chamada abaixo para usar o gabarito e a previsão corretos do conjunto de teste
print(classification_report(None, None))

## 12. Desafio extra opcional — SVM na tabela comparativa

Se você já completou a Atividade 4, volte até a tabela comparativa e garanta que a linha da SVM (`pred_svm`) também esteja incluída, usando a mesma métrica de acurácia. Se quiser ir além, compare também a precisão e a revocação da SVM na classe `virginica` usando o `classification_report`.

Depois, responda: considerando o cenário de alto custo de falso positivo em `virginica`, a SVM mudaria a sua escolha? Justifique com as métricas observadas.

*(Escreva sua resposta aqui, se for fazer o desafio extra.)*

## 13. Reflexão final

Escreva um parágrafo curto conectando os dois grandes temas desta lição: **métricas de avaliação** e **reprodutibilidade**. Por exemplo: por que faz sentido comparar modelos usando `classification_report` **e**, ao mesmo tempo, garantir que essa comparação seja feita com `random_state` fixo?

*(Escreva sua reflexão aqui.)*

## 14. Checklist de conclusão

* [ ] Abri o notebook no Google Colab.
* [ ] Salvei uma cópia no meu Google Drive.
* [ ] Executei as células de texto e código.
* [ ] Carreguei `X_treino.csv`, `X_teste.csv`, `y_treino.csv` e `y_teste.csv` gerados na M03-L02.
* [ ] Conferi a distribuição das classes no conjunto de teste.
* [ ] Gerei e interpretei a matriz de confusão de pelo menos três modelos.
* [ ] Gerei e interpretei o `classification_report` de pelo menos três modelos.
* [ ] Comparei os modelos em uma tabela e justifiquei minha escolha considerando métricas da classe `virginica`.
* [ ] Executei o experimento de variação de `random_state` e observei o efeito na acurácia.
* [ ] Corrigi o erro simples relacionado à avaliação de modelos.
* [ ] Preenchi a reflexão final.
* [ ] Finalizei o laboratório da lição.

## O que entregar

Mantenha este notebook salvo na sua cópia do Google Drive, com todas as células executadas em ordem e todas as células de Markdown de reflexão/diagnóstico preenchidas. Não é necessário exportar nenhum arquivo adicional — o notebook executado **é** o produto desta lição.